In [6]:
import pandas as pd
import numpy as np

# Load the dataset into a pandas DataFrame
df = pd.read_csv('Data_Cleaning_Task.csv')

# Display the first 5 rows to ensure it loaded correctly
df.head()

C:\Users\adila\AppData\Local\Temp\ipykernel_25356\2053808057.py:5: DtypeWarning: Columns (0: DefTwoPoint, 1: BlockingPlayer) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Data_Cleaning_Task.csv')


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


# --- DATA QUALITY REPORT ---

print("1. Dataset Shape (Total Rows, Total Columns):")
print(df.shape)

print("\n2. Total Duplicate Rows:")
print(df.duplicated().sum())

print("\n3. Top 20 Columns with the Most Missing (Null) Values:")
# We sort them to see the worst offenders first, since you have 102 columns!
print(df.isnull().sum().sort_values(ascending=False).head(20))

### Strategy for Handling Missing Data
Upon reviewing the Data Quality Report, several columns are missing over 80% of their values (e.g., `DefTwoPoint`, `BlockingPlayer`). 
**Decision:** I will delete any column that is missing more than 80% of its data, as imputing such a massive volume of nulls would introduce extreme bias. For the remaining columns with smaller amounts of missing data, I will use median imputation for numerical features and mode imputation for categorical features.

In [9]:
# Calculate the percentage of missing data for every column
missing_percentages = df.isnull().sum() / len(df) * 100

# Identify columns where the missing percentage is greater than 80%
columns_to_drop = missing_percentages[missing_percentages > 80].index

# Drop those columns from the DataFrame
df = df.drop(columns=columns_to_drop)

print(f"Successfully dropped {len(columns_to_drop)} columns.")
print(f"New dataset shape: {df.shape}")

Successfully dropped 18 columns.
New dataset shape: (362447, 84)


# Separate columns into numerical and categorical
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

# Fill missing numerical values with the median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with the mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Total missing values remaining in the entire dataset:")
print(df.isnull().sum().sum())

### Strategy for Handling Outliers
I will use the Interquartile Range (IQR) method to identify outliers in continuous numerical columns. Instead of deleting these rows and losing valuable data, I will *cap* the outliers at the upper and lower bounds (the 1.5 * IQR thresholds). This neutralizes extreme values without shrinking the dataset.

In [13]:
# Identify continuous numerical columns (ignoring binary flags with few unique values)
continuous_cols = [col for col in df.select_dtypes(include=['float64', 'int64']).columns if df[col].nunique() > 20]

def cap_outliers_iqr(dataframe, col_name):
    Q1 = dataframe[col_name].quantile(0.25)
    Q3 = dataframe[col_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap the values to the bounds
    dataframe[col_name] = np.where(dataframe[col_name] < lower_bound, lower_bound, dataframe[col_name])
    dataframe[col_name] = np.where(dataframe[col_name] > upper_bound, upper_bound, dataframe[col_name])

# Apply the capping function to the selected columns
for col in continuous_cols:
    cap_outliers_iqr(df, col)

print(f"Outliers successfully capped across {len(continuous_cols)} continuous columns.")


Outliers successfully capped across 35 continuous columns.


In [14]:
# Standardise the 'Date' column to actual datetime objects
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    print("Date column successfully converted to datetime format.")

# Produce the final summary required by the task checklist
print("\n--- FINAL DATA QUALITY SUMMARY ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]} (Reduced from 102)")
print(f"Total Missing Values: {df.isnull().sum().sum()} (Reduced from hundreds of thousands)")
print(f"Total Duplicate Rows: {df.duplicated().sum()}")

# Save the pristine dataset to a new CSV file
df.to_csv('cleaned_NFL_data.csv', index=False)
print("\nSuccess! Dataset exported as 'cleaned_NFL_data.csv'.")

Date column successfully converted to datetime format.

--- FINAL DATA QUALITY SUMMARY ---
Total Rows: 362447
Total Columns: 84 (Reduced from 102)
Total Missing Values: 0 (Reduced from hundreds of thousands)
Total Duplicate Rows: 0

Success! Dataset exported as 'cleaned_NFL_data.csv'.
